# Olist E-Commerce Project Starter

Use this starter notebook for the BAN 6003 final project option: **Olist Brazilian E-Commerce**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset is relational. Your main job is to preserve the row meaning of your final ABT.

- A common final ABT is **one row per order**, identified by `order_id`.
- Aggregate `order_items`, `order_payments`, and `order_reviews` to `order_id` before joining them to `orders`.
- Join `products`, category translations, and sellers to item-level data before aggregating those fields to order level.
- `customer_id` identifies the customer record attached to one order; `customer_unique_id` may repeat because one person can place multiple orders.
- Check row counts and duplicate `order_id` values after every major merge.
- Define targets only where the outcome is observable. An undelivered order is not automatically an on-time order, and a missing review is not a low review.
- State when a prediction would be made, then exclude fields that would not be available at that time.


## Setup

In [33]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [34]:
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_DIR / 'customers.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(DATA_DIR / 'order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'order_reviews.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(DATA_DIR / 'products.csv')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv')
geo = pd.read_csv(DATA_DIR / 'geolocation_zip_prefix.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation_zip_prefix': geo,
    'translations': translations,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [35]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


orders: 25,000 rows x 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24



customers: 25,000 rows x 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



items: 28,343 rows x 7 columns


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP



geolocation_zip_prefix: 10,009 rows x 6 columns


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25



translations: 71 rows x 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [36]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,orders,25000,8,0,1266
1,customers,25000,5,0,0
2,items,28343,7,0,0
3,payments,26154,5,0,0
4,reviews,24934,7,0,36618
5,products,13344,9,0,924
6,sellers,2234,4,0,0
7,geolocation_zip_prefix,10009,6,0,0
8,translations,71,2,0,0


### Key and Granularity Audit

Confirm the expected key for each table before planning joins. A duplicate `order_id` is expected in item, payment, and review detail tables because those tables can contain several records for one order. Their more detailed keys should still be checked. The provided geolocation table is already summarized to one row per zip-code prefix.


In [37]:
key_audit = pd.DataFrame([
    {"table": "orders", "expected_key": "order_id", "duplicate_keys": orders["order_id"].duplicated().sum(), "missing_key_values": orders["order_id"].isna().sum()},
    {"table": "customers", "expected_key": "customer_id", "duplicate_keys": customers["customer_id"].duplicated().sum(), "missing_key_values": customers["customer_id"].isna().sum()},
    {"table": "order_items", "expected_key": "order_id + order_item_id", "duplicate_keys": items.duplicated(["order_id", "order_item_id"]).sum(), "missing_key_values": items[["order_id", "order_item_id"]].isna().any(axis=1).sum()},
    {"table": "order_payments", "expected_key": "order_id + payment_sequential", "duplicate_keys": payments.duplicated(["order_id", "payment_sequential"]).sum(), "missing_key_values": payments[["order_id", "payment_sequential"]].isna().any(axis=1).sum()},
    {"table": "order_reviews", "expected_key": "review record (review_id may repeat)", "duplicate_keys": reviews.duplicated().sum(), "missing_key_values": reviews["order_id"].isna().sum()},
    {"table": "products", "expected_key": "product_id", "duplicate_keys": products["product_id"].duplicated().sum(), "missing_key_values": products["product_id"].isna().sum()},
    {"table": "sellers", "expected_key": "seller_id", "duplicate_keys": sellers["seller_id"].duplicated().sum(), "missing_key_values": sellers["seller_id"].isna().sum()},
    {"table": "geolocation_zip_prefix", "expected_key": "geolocation_zip_code_prefix", "duplicate_keys": geo["geolocation_zip_code_prefix"].duplicated().sum(), "missing_key_values": geo["geolocation_zip_code_prefix"].isna().sum()},
    {"table": "translations", "expected_key": "product_category_name", "duplicate_keys": translations["product_category_name"].duplicated().sum(), "missing_key_values": translations["product_category_name"].isna().sum()},
])

display(key_audit)
print(f"Item rows sharing an order_id: {items['order_id'].duplicated().sum():,}")
print(f"Payment rows sharing an order_id: {payments['order_id'].duplicated().sum():,}")
print(f"Review rows sharing an order_id: {reviews['order_id'].duplicated().sum():,}")
print(f"Repeated customer_unique_id values: {customers['customer_unique_id'].duplicated().sum():,}")


,table,expected_key,duplicate_keys,missing_key_values
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,order_items,order_id + order_item_id,0,0
3,order_payments,order_id + payment_sequential,0,0
4,order_reviews,review record (review_id may repeat),0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,geolocation_zip_prefix,geolocation_zip_code_prefix,0,0
8,translations,product_category_name,0,0


Item rows sharing an order_id: 3,552
Payment rows sharing an order_id: 1,154
Review rows sharing an order_id: 137
Repeated customer_unique_id values: 236


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Olist notes:

- Keep a table inventory: what does one row mean in each file?
- `order_items`, `order_payments`, and `order_reviews` are not guaranteed to be one row per order.
- Missing review comments are common and should be described, not automatically deleted.
- Product categories may need translation using `product_category_name_translation.csv`.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:
- Why this question matters:
- Tables/files used:
- Row meaning of each important table:
- Early data quality issues:
- Initial cleaning plan:
- Possible target/outcome:
- Ethics, privacy, or governance concerns:



In [38]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.



In [39]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


orders


,dtype
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]
order_estimated_delivery_date,datetime64[us]


,missing_rate
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
order_approved_at,0.00156
order_id,0.00000
order_purchase_timestamp,0.00000
order_status,0.00000
customer_id,0.00000
order_estimated_delivery_date,0.00000



customers


,dtype
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str


...

,dtype
geolocation_zip_code_prefix,int64
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,str
geolocation_state,str
geolocation_records,int64


,missing_rate
geolocation_zip_code_prefix,0.0
geolocation_lat,0.0
geolocation_lng,0.0
geolocation_city,0.0
geolocation_state,0.0
geolocation_records,0.0



translations


,dtype
product_category_name,str
product_category_name_english,str


,missing_rate
product_category_name,0.0
product_category_name_english,0.0


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what key defines one row, and which tables were joined or aggregated.

Critical integration reminders:

- Enrich item records with product, category, and seller attributes while they are still item-level.
- Aggregate item, payment, and review records separately to `order_id`.
- Join each order-level summary to `orders` with merge validation.
- Treat the order-level ABT as trustworthy only after its row count and unique key have been checked.

The starter below demonstrates this sequence. Revise the variables to fit your question, and explain why each retained variable is useful.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [40]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.



In [41]:
products_enriched = products.merge(
    translations,
    on="product_category_name",
    how="left",
    validate="many_to_one",
)

item_detail = (
    items
    .merge(
        products_enriched[[
            "product_id", "product_category_name_english", "product_weight_g"
        ]],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        sellers[["seller_id", "seller_state"]],
        on="seller_id",
        how="left",
        validate="many_to_one",
    )
)

item_summary = (
    item_detail.groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        category_count=("product_category_name_english", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        item_price_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        product_weight_mean_g=("product_weight_g", "mean"),
    )
    .reset_index()
)

payment_summary = (
    payments.groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "count"),
        payment_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_count=("payment_type", "nunique"),
    )
    .reset_index()
)

reviews_for_summary = reviews.assign(
    review_comment_present=reviews["review_comment_message"].notna().astype(int)
)
review_summary = (
    reviews_for_summary.groupby("order_id")
    .agg(
        review_record_count=("review_id", "size"),
        review_score_mean=("review_score", "mean"),
        review_comment_present=("review_comment_present", "max"),
    )
    .reset_index()
)

customer_geo = customers.merge(
    geo[[
        "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
        "geolocation_records"
    ]],
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng",
    "geolocation_records": "customer_geo_source_records",
})

abt = (
    orders
    .merge(customer_geo, on="customer_id", how="left", validate="one_to_one")
    .merge(item_summary, on="order_id", how="left", validate="one_to_one")
    .merge(payment_summary, on="order_id", how="left", validate="one_to_one")
    .merge(review_summary, on="order_id", how="left", validate="one_to_one")
)

abt["purchase_month"] = abt["order_purchase_timestamp"].dt.month
abt["purchase_dayofweek"] = abt["order_purchase_timestamp"].dt.dayofweek
abt["promised_delivery_days"] = (
    abt["order_estimated_delivery_date"] - abt["order_purchase_timestamp"]
).dt.days
abt["delivery_days"] = (
    abt["order_delivered_customer_date"] - abt["order_purchase_timestamp"]
).dt.days

delivery_outcome_observed = (
    abt["order_status"].eq("delivered")
    & abt["order_delivered_customer_date"].notna()
    & abt["order_estimated_delivery_date"].notna()
)
abt["delivered_late"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
abt.loc[delivery_outcome_observed, "delivered_late"] = (
    abt.loc[delivery_outcome_observed, "order_delivered_customer_date"]
    > abt.loc[delivery_outcome_observed, "order_estimated_delivery_date"]
).astype(int)

abt["high_review_score"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
review_outcome_observed = abt["review_score_mean"].notna()
abt.loc[review_outcome_observed, "high_review_score"] = (
    abt.loc[review_outcome_observed, "review_score_mean"] >= 4
).astype(int)


In [42]:
abt_validation = pd.Series({
    "orders_before_merges": len(orders),
    "abt_rows_after_merges": len(abt),
    "unique_order_ids": abt["order_id"].nunique(),
    "duplicate_order_ids": abt["order_id"].duplicated().sum(),
    "orders_with_item_summary": abt["order_item_count"].notna().sum(),
    "orders_with_payment_summary": abt["payment_total"].notna().sum(),
    "orders_with_review_summary": abt["review_score_mean"].notna().sum(),
    "orders_with_observed_delivery_target": abt["delivered_late"].notna().sum(),
    "orders_with_observed_review_target": abt["high_review_score"].notna().sum(),
})

display(abt_validation.to_frame("value"))
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


,value
orders_before_merges,25000
abt_rows_after_merges,25000
unique_order_ids,25000
duplicate_order_ids,0
orders_with_item_summary,24791
orders_with_payment_summary,25000
orders_with_review_summary,24797
orders_with_observed_delivery_target,24222
orders_with_observed_review_target,24797


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,customer_lat,customer_lng,customer_geo_source_records,order_item_count,product_count,seller_count,category_count,seller_state_count,item_price_total,freight_total,product_weight_mean_g,payment_record_count,payment_total,payment_installments_max,payment_type_count,review_record_count,review_score_mean,review_comment_present,purchase_month,purchase_dayofweek,promised_delivery_days,delivery_days,delivered_late,high_review_score
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27,32ea3bdedab835c3aa6cb68ce66565ef,4106,sao paulo,SP,4106.0,-23.581451,-46.635029,45.0,1.0,1.0,1.0,1.0,1.0,29.90,15.56,300.0,1,45.46,1,1,1.0,4.0,1.0,10,0,23,23.0,0,1
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23,87776adb449c551e74c13fc34f036105,12030,taubate,SP,12030.0,-23.032142,-45.570461,218.0,1.0,1.0,1.0,1.0,1.0,29.99,10.96,9000.0,1,40.95,4,1,1.0,5.0,0.0,10,0,50,10.0,0,1
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24,823c47d4abda1f8ce7568145f76c2b85,13175,sumare,SP,13175.0,-22.817992,-47.241101,173.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,50.74,3,1,1.0,2.0,0.0,10,1,50,NaN,<NA>,0
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28,0829f7df6577d5a4b65439bea701405f,31210,belo horizonte,MG,31210.0,-19.906712,-43.948897,156.0,1.0,1.0,1.0,1.0,1.0,249.90,17.59,610.0,1,267.49,10,1,1.0,3.0,0.0,10,1,54,12.0,0,0
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28,df2988ba3ed226b10521a0e4da849b61,23092,rio de janeiro,RJ,23092.0,-22.870856,-43.543849,217.0,1.0,1.0,1.0,1.0,1.0,67.90,18.98,2100.0,7,86.88,1,1,1.0,5.0,1.0,10,1,54,51.0,0,1


,missing_rate
delivered_late,0.03112
order_delivered_customer_date,0.03100
delivery_days,0.03100
order_delivered_carrier_date,0.01808
product_weight_mean_g,0.00844
item_price_total,0.00836
seller_count,0.00836
order_item_count,0.00836
freight_total,0.00836
category_count,0.00836


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the order-level ABT row count, key uniqueness, and target eligibility without creating another permanent database file.


In [43]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("olist_order_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        '''
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT order_id) AS unique_order_ids,
            SUM(CASE WHEN delivered_late IS NOT NULL THEN 1 ELSE 0 END) AS delivery_target_rows,
            SUM(CASE WHEN high_review_score IS NOT NULL THEN 1 ELSE 0 END) AS review_target_rows
        FROM olist_order_abt
        ''',
        connection,
    )

display(sql_abt_check)


,abt_rows,unique_order_ids,delivery_target_rows,review_target_rows
0,25000,25000,24222,24797


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [44]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,order_id,str,,,,
1,customer_id,str,,,,
2,order_status,str,,,,
3,order_purchase_timestamp,datetime64[us],,,,
4,order_approved_at,datetime64[us],,,,


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminders:

- If you model `delivered_late`, use only rows where the nullable target is defined; undelivered orders do not reveal a normal late/on-time outcome.
- For a prediction made at purchase time, do not use actual delivery duration, final order status, review fields, or other post-purchase outcomes as predictors.
- If you model `high_review_score`, use only rows with an observed review and do not use review-derived fields as predictors.
- Check class balance, keep a reproducible train/test split, and interpret precision and recall when the positive class is uncommon.
- Explain the intended decision and prediction timing before choosing features.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [45]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [46]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,25000,25000,3b697a20d9e427646d92567910af6d57,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,25000,25000,355077684019f7f60a031656bd7262b8,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,25000,8,delivered,24222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,25000,NaN,NaN,NaN,2017-12-31 22:18:29.561480,2016-10-03 09:44:50,2017-09-13 11:52:50,2018-01-19 19:08:12.500000,2018-05-05 11:17:57,2018-10-01 15:30:09,NaN
order_approved_at,24961,NaN,NaN,NaN,2018-01-01 06:46:13.413765,2016-10-04 10:19:23,2017-09-13 19:55:11,2018-01-20 09:08:59,2018-05-05 11:50:40,2018-09-03 17:40:06,NaN
order_delivered_carrier_date,24548,NaN,NaN,NaN,2018-01-05 06:50:25.146529,2016-10-08 10:34:01,2017-09-18 18:01:51.750000,2018-01-24 20:01:44.500000,2018-05-08 14:21:00,2018-09-04 15:25:00,NaN
order_delivered_customer_date,24225,NaN,NaN,NaN,2018-01-14 19:15:33.384024,2016-10-11 14:46:49,2017-09-26 19:04:55,2018-02-03 03:49:26,2018-05-16 12:26:50,2018-09-27 02:24:33,NaN
order_estimated_delivery_date,25000,NaN,NaN,NaN,2018-01-24 17:19:26.976000,2016-10-27 00:00:00,2017-10-04 00:00:00,2018-02-15 00:00:00,2018-05-28 00:00:00,2018-10-23 00:00:00,NaN
customer_unique_id,25000,24764,8d50f5eadf50201ccdcedfb9e2ac8455,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,25000.0,NaN,NaN,NaN,34966.35068,1004.0,11431.75,24326.0,58037.0,99970.0,29706.603087


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [47]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "olist_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "olist_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "olist_model_metrics.csv"
# report_path = REPORTS_DIR / "olist_final_report.html"


In [48]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "olist_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "olist_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
